In [2]:
import os
import json
from dotenv import load_dotenv

In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.callbacks.manager import get_openai_callback

In [4]:
load_dotenv()
key = os.getenv("GROQ_API_KEY")

In [5]:
openaiModel =  ChatGroq(model="openai/gpt-oss-120b",temperature = 0.3)

In [6]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [7]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [8]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [9]:
TEMPLATE1="""
You are an expert in {subject}. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
Quiz_MCQs:
{quiz}

Check from an subject expert for the above quiz:
"""

In [10]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE1)

In [11]:
quiz_chain = (
    quiz_generation_prompt
    | openaiModel
    | StrOutputParser()
)

In [12]:
review_chain = (
    quiz_evaluation_prompt
    | openaiModel
    | StrOutputParser()
)

In [13]:
generate_evaluate_chain = (
    RunnablePassthrough.assign(
        quiz=quiz_chain
    )
    .assign(
        review=(
            lambda x: {
                "quiz": x["quiz"],
                "subject": x["subject"],
            }
        )
        | review_chain
    )
)

In [14]:
from pathlib import Path
file_path=r"D:\Projects\mcqgen\experiment\test.txt"

In [15]:
file_path

'D:\\Projects\\mcqgen\\experiment\\test.txt'

In [16]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [17]:
print(TEXT)

Long Short-Term Memory (LSTM) is a specialized type of Recurrent Neural Network (RNN) designed to process sequential data, such as time-series forecasting, speech recognition, and language translation. Unlike traditional networks, LSTMs are engineered to remember vital information over long periods, making them incredibly effective at understanding context and predicting future steps based on past inputs.The secret to an LSTM's power lies in its unique internal structure, which replaces the repeating modules of standard RNNs with a cell state and a series of "gates". The cell state acts as a conveyor belt, carrying information straight through the entire chain of processing. The gatesâ€”specifically the forget gate, input gate, and output gateâ€”are small neural networks that act as valves. They determine exactly which pieces of information are allowed onto the conveyor belt to be stored, which are updated, and which are discarded as noise.These gating mechanisms allow LSTMs to solve t

In [18]:
NUMBER=5 
SUBJECT="Deep Learning"
TONE="simple"

In [19]:
#setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain.invoke(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )

In [20]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:2576
Prompt Tokens:1142
Completion Tokens:1434
Total Cost:0.0


In [21]:
response

{'text': 'Long Short-Term Memory (LSTM) is a specialized type of Recurrent Neural Network (RNN) designed to process sequential data, such as time-series forecasting, speech recognition, and language translation. Unlike traditional networks, LSTMs are engineered to remember vital information over long periods, making them incredibly effective at understanding context and predicting future steps based on past inputs.The secret to an LSTM\'s power lies in its unique internal structure, which replaces the repeating modules of standard RNNs with a cell state and a series of "gates". The cell state acts as a conveyor belt, carrying information straight through the entire chain of processing. The gatesâ€”specifically the forget gate, input gate, and output gateâ€”are small neural networks that act as valves. They determine exactly which pieces of information are allowed onto the conveyor belt to be stored, which are updated, and which are discarded as noise.These gating mechanisms allow LSTMs

In [24]:
quiz_str=response.get("quiz")
quiz_dict = json.loads(quiz_str)
quiz_dict

{'1': {'mcq': 'What kind of neural network is an LSTM?',
  'options': {'a': 'Convolutional Neural Network (CNN)',
   'b': 'Recurrent Neural Network (RNN)',
   'c': 'Generative Adversarial Network (GAN)',
   'd': 'Support Vector Machine (SVM)'},
  'correct': 'b'},
 '2': {'mcq': 'Which training problem do LSTMs help to solve?',
  'options': {'a': 'Exploding gradient',
   'b': 'Vanishing gradient',
   'c': 'Overfitting',
   'd': 'Underfitting'},
  'correct': 'b'},
 '3': {'mcq': 'What are the three gates inside an LSTM cell?',
  'options': {'a': 'Reset gate, update gate, candidate gate',
   'b': 'Forget gate, input gate, output gate',
   'c': 'Attention gate, memory gate, loss gate',
   'd': 'Filter gate, merge gate, split gate'},
  'correct': 'b'},
 '4': {'mcq': 'In the LSTM architecture, the cell state is often described as:',
  'options': {'a': 'A filter that removes noise',
   'b': 'A conveyor belt that carries information',
   'c': 'An activation function',
   'd': 'A dropout layer'},

In [25]:
quiz_table_data = []
for key, value in quiz_dict.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [26]:
import pandas as pd

In [27]:
df=pd.DataFrame(quiz_table_data)

In [28]:
df

,MCQ,Choices,Correct
0,What kind of neural network is an LSTM?,a: Convolutional Neural Network (CNN) | b: Rec...,b
1,Which training problem do LSTMs help to solve?,a: Exploding gradient | b: Vanishing gradient ...,b
2,What are the three gates inside an LSTM cell?,"a: Reset gate, update gate, candidate gate | b...",b
3,"In the LSTM architecture, the cell state is of...",a: A filter that removes noise | b: A conveyor...,b
4,Which of the following applications are mentio...,"a: Predictive text, stock analysis, and voice ...",a
